# Notebook 2 — Feature Engineering


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../health_lifestyle_dataset.csv')
print(f'Shape initiale: {df.shape}')

Shape initiale: (100000, 16)


## 1. Encodage de la variable gender

In [2]:
df['gender_enc'] = (df['gender'] == 'Male').astype(int)
print(df[['gender','gender_enc']].drop_duplicates())

   gender  gender_enc
0    Male           1
1  Female           0


## 2. Création de nouvelles variables

In [3]:
# Hypertension (seuils ESC 2018)
df['hypertension'] = ((df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90)).astype(int)
print(f'Hypertension: {df["hypertension"].mean()*100:.1f}% des individus')

# Catégories IMC (OMS)
df['bmi_cat'] = pd.cut(df['bmi'], bins=[0,18.5,25,30,100], labels=[0,1,2,3]).astype(int)
print('\nCatégories IMC:')
print(df['bmi_cat'].value_counts().sort_index())

# Score de style de vie
df['lifestyle_score'] = (
    df['daily_steps']/df['daily_steps'].max() +
    df['sleep_hours']/df['sleep_hours'].max() +
    df['water_intake_l']/df['water_intake_l'].max() -
    df['smoker'] - df['alcohol'] -
    df['bmi']/df['bmi'].max() -
    df['calories_consumed']/df['calories_consumed'].max()
)
print(f'\nlifestyle_score: mean={df["lifestyle_score"].mean():.3f}, std={df["lifestyle_score"].std():.3f}')

Hypertension: 72.3% des individus

Catégories IMC:
bmi_cat
0     2427
1    29425
2    22739
3    45409
Name: count, dtype: int64

lifestyle_score: mean=-0.154, std=0.789


## 3. Standardisation et normalisation

In [4]:
continuous = ['age','bmi','daily_steps','sleep_hours','water_intake_l',
              'calories_consumed','resting_hr','systolic_bp','diastolic_bp','cholesterol']

# Standardisation z-score
scaler = StandardScaler()
df_std = pd.DataFrame(scaler.fit_transform(df[continuous]), columns=continuous)
print('Après standardisation (mean≈0, std≈1):')
print(df_std.describe().loc[['mean','std']].round(3))

# Normalisation min-max
df_mm = (df[continuous] - df[continuous].min()) / (df[continuous].max() - df[continuous].min())
print('\nAprès normalisation min-max (min=0, max=1):')
print(df_mm.describe().loc[['min','max']].round(3))

Après standardisation (mean≈0, std≈1):
      age  bmi  daily_steps  sleep_hours  water_intake_l  calories_consumed  \
mean -0.0  0.0         -0.0          0.0             0.0                0.0   
std   1.0  1.0          1.0          1.0             1.0                1.0   

      resting_hr  systolic_bp  diastolic_bp  cholesterol  
mean         0.0         -0.0           0.0         -0.0  
std          1.0          1.0           1.0          1.0  

Après normalisation min-max (min=0, max=1):
     age  bmi  daily_steps  sleep_hours  water_intake_l  calories_consumed  \
min  0.0  0.0          0.0          0.0             0.0                0.0   
max  1.0  1.0          1.0          1.0             1.0                1.0   

     resting_hr  systolic_bp  diastolic_bp  cholesterol  
min         0.0          0.0           0.0          0.0  
max         1.0          1.0           1.0          1.0  


## 4. Dataset final

In [5]:
features = ['age','bmi','daily_steps','sleep_hours','water_intake_l','calories_consumed',
            'smoker','alcohol','resting_hr','systolic_bp','diastolic_bp',
            'family_history','gender_enc','bmi_cat','hypertension','lifestyle_score']
print(f'Nombre de features: {len(features)}')
print('Variables:', features)
df[features + ['disease_risk']].to_csv('../data_processed.csv', index=False)
print('\nDataset sauvegardé dans data_processed.csv')

Nombre de features: 16
Variables: ['age', 'bmi', 'daily_steps', 'sleep_hours', 'water_intake_l', 'calories_consumed', 'smoker', 'alcohol', 'resting_hr', 'systolic_bp', 'diastolic_bp', 'family_history', 'gender_enc', 'bmi_cat', 'hypertension', 'lifestyle_score']

Dataset sauvegardé dans data_processed.csv
